In [3]:
#r "nuget:ScottPlot,5.0.21"
#r "task14\bin\Debug\net10.0\task14.dll"

using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.IO;
using System.Linq;
using ScottPlot;
using task14;

double a = -100, b = 100;
double targetPrecision = 1e-4;
int warmupRuns = 200;
int measureRuns = 20;
int maxThreads = Environment.ProcessorCount * 2;

Func<double, double> f = Math.Sin;
double exactValue = 0.0;

double[] stepCandidates = { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };

double optimalStep = 0;
int optimalThreadsForStep = 1;
double bestParallelTimeForOptimalStep = 0;
double bestSingleTimeForOptimalStep = 0;
double bestImprovement = 0;
bool foundOptimalStep = false;

var allStepData = new List<(double step, int threads, double singleTime, double parallelTime, double improvement)>();
var threadTimes = new List<(int threads, double time)>();

foreach (var step in stepCandidates)
{
    double val = OneThread.Solve(a, b, f, step);
    double error = Math.Abs(val - exactValue);
    
    if (error > targetPrecision)
    {
        Console.WriteLine($"Шаг {step:E1}: погрешность {error:E3} > {targetPrecision:E1}, не подходит");
        continue;
    }

    var swSingle = new Stopwatch();
    double singleTotal = 0;
    for (int run = 0; run < warmupRuns + measureRuns; run++)
    {
        swSingle.Restart();
        OneThread.Solve(a, b, f, step);
        swSingle.Stop();
        if (run >= warmupRuns) singleTotal += swSingle.Elapsed.TotalMilliseconds;
    }
    double singleTime = singleTotal / measureRuns;
    
    Console.WriteLine($"\nШаг {step:E1}: точность {error:E5}, 1 поток: {singleTime:F3} мс");
    
    double bestParallelTime = double.MaxValue;
    int bestThreads = 1;
    
    var currentStepTimes = new List<(int threads, double time)>();
    currentStepTimes.Add((1, singleTime));
    
    for (int threads = 2; threads <= maxThreads; threads += 2)
    {
        var sw = new Stopwatch();
        double totalTime = 0;
        
        for (int run = 0; run < warmupRuns + measureRuns; run++)
        {
            sw.Restart();
            DefiniteIntegral.Solve(a, b, f, step, threads);
            sw.Stop();
            if (run >= warmupRuns) totalTime += sw.Elapsed.TotalMilliseconds;
        }
        
        double avgTime = totalTime / measureRuns;
        currentStepTimes.Add((threads, avgTime));
        
        if (avgTime < bestParallelTime)
        {
            bestParallelTime = avgTime;
            bestThreads = threads;
        }
    }
    
    double improvement = (singleTime - bestParallelTime) / singleTime * 100;
    
    allStepData.Add((step, bestThreads, singleTime, bestParallelTime, improvement));
    
    Console.WriteLine($"  Лучше всего: {bestThreads} потоков, {bestParallelTime:F3} мс (ускорение {improvement:F1}%)");
    
    optimalStep = step;
    optimalThreadsForStep = bestThreads;
    bestParallelTimeForOptimalStep = bestParallelTime;
    bestSingleTimeForOptimalStep = singleTime;
    bestImprovement = improvement;
    threadTimes = currentStepTimes;
    
    if (improvement >= 15)
    {
        Console.WriteLine($"  Ускорение больше 15%, останавливаемся");
        foundOptimalStep = true;
        break;
    }
}

if (!foundOptimalStep)
{
    Console.WriteLine($"\nУскорение 15% не достигнуто. Выбран шаг {optimalStep:E1} (ускорение {bestImprovement:F1}%)");
}

var plt = new ScottPlot.Plot();

double[] plotX = threadTimes.Select(t => t.time).ToArray();
double[] plotY = threadTimes.Select(t => (double)t.threads).ToArray();

var scatter = plt.Add.Scatter(plotX, plotY);
scatter.LineWidth = 3;
scatter.MarkerSize = 10;
scatter.Color = ScottPlot.Color.FromHex("#1e3a8a");

plt.Title($"Время от числа потоков (шаг {optimalStep:E1})");
plt.XLabel("Время, мс");
plt.YLabel("Потоки");

string fileName = $"plot_{optimalStep:E1}.png";
plt.SavePng(fileName, 700, 500);
Console.WriteLine($"\nГрафик: {fileName}");

Console.WriteLine($"\nИтог: шаг {optimalStep:E1}, потоков {optimalThreadsForStep}");
Console.WriteLine($"  1 поток: {bestSingleTimeForOptimalStep:F3} мс");
Console.WriteLine($"  {optimalThreadsForStep} потоков: {bestParallelTimeForOptimalStep:F3} мс");
Console.WriteLine($"  Ускорение: {bestImprovement:F1}%");

string resultFilePath = "results.txt";
string results = 
$@"Результаты
Шаг: {optimalStep:E1}
Потоков: {optimalThreadsForStep}
Время (1): {bestSingleTimeForOptimalStep:F3} мс
Время ({optimalThreadsForStep}): {bestParallelTimeForOptimalStep:F3} мс
Ускорение: {bestImprovement:F1}%
";

File.WriteAllText(resultFilePath, results);
Console.WriteLine($"Результаты в файле: {resultFilePath}");

Installed Packages ScottPlot, 5.0.21


Шаг 1.0E-001: точность 5.04458E-015, 1 поток: 0.094 мс
  Лучше всего: 2 потоков, 0.253 мс (ускорение -168.5%)

Шаг 1.0E-002: точность 1.49195E-014, 1 поток: 0.485 мс
  Лучше всего: 2 потоков, 0.618 мс (ускорение -27.3%)

Шаг 1.0E-003: точность 2.34296E-016, 1 поток: 5.944 мс
  Лучше всего: 6 потоков, 2.798 мс (ускорение 52.9%)
  Ускорение больше 15%, останавливаемся

График: plot_1.0E-003.png

Итог: шаг 1.0E-003, потоков 6
  1 поток: 5.944 мс
  6 потоков: 2.798 мс
  Ускорение: 52.9%
Результаты в файле: results.txt
